# Track 2: Medical Desert Planner - Data Pipeline

## Objective
Build a trust-weighted facility aggregation system to identify real healthcare gaps vs data-poor regions across India.

## Dataset Overview
- **Source**: 10,000 Indian healthcare facilities
- **Key Fields**: location, specialties, capability, procedure, equipment, description
- **Geographic**: state, city, pincode, lat/long
- **Quality**: Noisy, uneven coverage across evidence fields

## Pipeline Steps
1. **Data Exploration**: Assess quality and coverage
2. **Data Cleaning**: Parse arrays, fix types, validate coordinates, handle nulls
3. **Data Enrichment**: Join with pincode directory and district health indicators
4. **Trust Scoring**: Weight facilities by data completeness and quality
5. **Geographic Aggregation**: Rollup by state/district/city/pincode
6. **Gap Analysis**: Distinguish real gaps from data-poor regions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

# Load the main facilities dataset
facilities_raw = spark.table("databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities")

print(f"Total facilities: {facilities_raw.count():,}")
print(f"Total columns: {len(facilities_raw.columns)}")

# Display sample
display(facilities_raw.limit(5))

In [0]:
# Assess data completeness for key fields
key_fields = [
    'name', 'address_stateOrRegion', 'address_city', 'address_zipOrPostcode',
    'latitude', 'longitude', 'specialties', 'capability', 'procedure', 
    'equipment', 'description', 'yearEstablished', 'numberDoctors', 'capacity'
]

# Build completeness expressions - handle numeric fields separately
completeness_exprs = []
for c in key_fields:
    if c in ['latitude', 'longitude']:
        # Numeric fields - just check for not null
        expr = (F.count(F.when(F.col(c).isNotNull(), c)) / F.count('*') * 100).alias(c)
    else:
        # String fields - check for not null, not empty, not '[]'
        expr = (F.count(F.when(
            F.col(c).isNotNull() & 
            (F.col(c) != '') & 
            (F.col(c) != '[]') & 
            (F.col(c) != 'null'), c
        )) / F.count('*') * 100).alias(c)
    completeness_exprs.append(expr)

completeness = facilities_raw.select(completeness_exprs).collect()[0]

# Create summary DataFrame
completeness_df = spark.createDataFrame(
    [(field, round(completeness[field], 1)) for field in key_fields],
    ['field', 'completeness_pct']
).orderBy(F.desc('completeness_pct'))

print("Data Completeness by Field:")
display(completeness_df)

In [0]:
# Analyze geographic distribution
geo_summary = facilities_raw.groupBy('address_stateOrRegion').agg(
    F.count('*').alias('facility_count'),
    F.countDistinct('address_city').alias('cities'),
    F.countDistinct('address_zipOrPostcode').alias('pincodes'),
    F.sum(F.when(F.col('latitude').isNotNull(), 1).otherwise(0)).alias('with_coordinates'),
    F.sum(F.when(F.col('capability').isNotNull() & (F.col('capability') != '[]'), 1).otherwise(0)).alias('with_capability'),
    F.sum(F.when(F.col('equipment').isNotNull() & (F.col('equipment') != '[]'), 1).otherwise(0)).alias('with_equipment')
).orderBy(F.desc('facility_count'))

print("Facilities by State:")
display(geo_summary)

In [0]:
# Clean and transform the raw data
facilities_cleaned = facilities_raw.select(
    'unique_id',
    'name',
    F.trim(F.col('address_stateOrRegion')).alias('state'),
    F.trim(F.col('address_city')).alias('city'),
    F.trim(F.col('address_zipOrPostcode')).alias('pincode'),
    
    # Validate and clean coordinates (India bounds: lat 8-35, lon 68-97)
    F.when(
        (F.col('latitude').between(8, 35)) & (F.col('longitude').between(68, 97)),
        F.col('latitude')
    ).alias('latitude'),
    F.when(
        (F.col('latitude').between(8, 35)) & (F.col('longitude').between(68, 97)),
        F.col('longitude')
    ).alias('longitude'),
    
    # Parse array-like string fields - handle both null and '[]'
    F.when(
        F.col('specialties').isNotNull() & (F.col('specialties') != '[]'),
        F.from_json(F.col('specialties'), ArrayType(StringType()))
    ).alias('specialties'),
    
    F.when(
        F.col('capability').isNotNull() & (F.col('capability') != '[]') & (F.col('capability') != 'null'),
        F.from_json(F.col('capability'), ArrayType(StringType()))
    ).alias('capability'),
    
    F.when(
        F.col('equipment').isNotNull() & (F.col('equipment') != '[]'),
        F.from_json(F.col('equipment'), ArrayType(StringType()))
    ).alias('equipment'),
    
    F.when(
        F.col('procedure').isNotNull() & (F.col('procedure') != '[]'),
        F.from_json(F.col('procedure'), ArrayType(StringType()))
    ).alias('procedure'),
    
    'description',
    
    # Convert numeric fields - check if matches integer pattern before casting
    F.when(
        (F.col('yearEstablished').isNotNull()) & 
        (F.col('yearEstablished').rlike('^[0-9]+$')),
        F.col('yearEstablished').cast('int')
    ).alias('year_established'),
    
    F.when(
        (F.col('numberDoctors').isNotNull()) & 
        (F.col('numberDoctors').rlike('^[0-9]+$')),
        F.col('numberDoctors').cast('int')
    ).alias('number_doctors'),
    
    F.when(
        (F.col('capacity').isNotNull()) & 
        (F.col('capacity').rlike('^[0-9]+$')),
        F.col('capacity').cast('int')
    ).alias('capacity'),
    
    # Additional metadata fields
    'facilityTypeId',
    'source',
    'source_urls'
)

print(f"Cleaned facilities: {facilities_cleaned.count():,}")
print(f"Facilities with valid coordinates: {facilities_cleaned.filter(F.col('latitude').isNotNull()).count():,}")
display(facilities_cleaned.limit(5))

In [0]:
# Load India Post pincode directory for district mapping
pincode_dir = spark.table("databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory")

# Get unique pincode-district-state mappings
pincode_lookup = pincode_dir.select(
    F.col('pincode').cast('string').alias('pincode'),
    F.initcap(F.trim(F.col('district'))).alias('district'),
    F.initcap(F.trim(F.col('statename'))).alias('state_name')
).dropDuplicates(['pincode'])

print(f"Unique pincodes in directory: {pincode_lookup.count():,}")

# Join with facilities
facilities_enriched = facilities_cleaned.join(
    pincode_lookup,
    facilities_cleaned.pincode == pincode_lookup.pincode,
    'left'
).drop(pincode_lookup.pincode)

# Use pincode directory state name if facility state is missing
facilities_enriched = facilities_enriched.withColumn(
    'state',
    F.coalesce(F.col('state'), F.col('state_name'))
)

print(f"Facilities with district mapping: {facilities_enriched.filter(F.col('district').isNotNull()).count():,}")
display(facilities_enriched.select('unique_id', 'name', 'state', 'city', 'district', 'pincode').limit(10))

In [0]:
# Load NFHS-5 district health indicators
nfhs = spark.table("databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators")

# Select key health indicators
health_indicators = nfhs.select(
    F.initcap(F.trim(F.col('district_name'))).alias('district'),
    F.initcap(F.trim(F.col('state_ut'))).alias('state_ut'),
    'institutional_birth_5y_pct',
    'child_12_23m_fully_vaccinated_based_on_information_from_eit_pct',
    'women_age_15_49_with_10_or_more_years_of_schooling_pct',
    'hh_member_covered_health_insurance_pct',
    'households_surveyed'
)

print(f"Districts with health indicators: {health_indicators.count():,}")

# Join with facilities (many-to-one: many facilities to one district)
facilities_with_health = facilities_enriched.join(
    health_indicators,
    (facilities_enriched.district == health_indicators.district) & 
    (facilities_enriched.state == health_indicators.state_ut),
    'left'
).drop(health_indicators.district).drop(health_indicators.state_ut)

print(f"Facilities with health indicators: {facilities_with_health.filter(F.col('institutional_birth_5y_pct').isNotNull()).count():,}")
display(facilities_with_health.limit(5))

In [0]:
# Calculate trust scores based on data completeness and quality
# Scale: 0-3 (0=no evidence, 1=weak, 2=moderate, 3=strong)

facilities_with_trust = facilities_with_health.withColumn(
    # Data completeness indicators (0 or 1 for each)
    'has_coordinates', F.when(F.col('latitude').isNotNull(), 1).otherwise(0)
).withColumn(
    'has_specialties', F.when(F.size(F.col('specialties')) > 0, 1).otherwise(0)
).withColumn(
    'has_capability', F.when(F.size(F.col('capability')) > 0, 1).otherwise(0)
).withColumn(
    'has_equipment', F.when(F.size(F.col('equipment')) > 0, 1).otherwise(0)
).withColumn(
    'has_procedure', F.when(F.size(F.col('procedure')) > 0, 1).otherwise(0)
).withColumn(
    'has_description', F.when(F.col('description').isNotNull() & (F.length(F.col('description')) > 50), 1).otherwise(0)
).withColumn(
    'has_capacity', F.when(F.col('capacity').isNotNull(), 1).otherwise(0)
).withColumn(
    'has_doctors', F.when(F.col('number_doctors').isNotNull(), 1).otherwise(0)
).withColumn(
    'has_year', F.when(F.col('year_established').isNotNull(), 1).otherwise(0)
).withColumn(
    # Overall data completeness score (0-9)
    'data_completeness_score',
    F.col('has_coordinates') + 
    F.col('has_specialties') + 
    F.col('has_capability') + 
    F.col('has_equipment') + 
    F.col('has_procedure') + 
    F.col('has_description') + 
    F.col('has_capacity') + 
    F.col('has_doctors') + 
    F.col('has_year')
).withColumn(
    # Convert to trust level (0-3 scale)
    'overall_trust_score',
    F.when(F.col('data_completeness_score') >= 7, 3)  # Strong: 7-9 fields
     .when(F.col('data_completeness_score') >= 5, 2)  # Moderate: 5-6 fields
     .when(F.col('data_completeness_score') >= 3, 1)  # Weak: 3-4 fields
     .otherwise(0)  # None: 0-2 fields
)

print("Trust Score Distribution:")
trust_dist = facilities_with_trust.groupBy('overall_trust_score').agg(
    F.count('*').alias('facility_count')
).orderBy('overall_trust_score')
display(trust_dist)

# Show example facilities by trust level
print("\nSample facilities by trust level:")
display(
    facilities_with_trust
    .select('unique_id', 'name', 'state', 'city', 'district', 'overall_trust_score', 'data_completeness_score')
    .orderBy(F.desc('overall_trust_score'), F.desc('data_completeness_score'))
    .limit(10)
)

In [0]:
# Define key capabilities for Medical Desert analysis
key_capabilities = [
    'emergency', 'maternity', 'surgery', 'icu', 'dialysis', 
    'nicu', 'oncology', 'trauma', 'blood_bank', 'ambulance'
]

# Function to check if capability is mentioned in evidence fields
def create_capability_check(capability_name, keywords):
    """Create a column that checks if capability is mentioned in evidence fields"""
    # Check in capability array
    capability_check = F.expr(f"exists(capability, x -> lower(x) rlike '{"|.join(keywords)}')").cast('int')
    
    # Check in equipment array
    equipment_check = F.expr(f"exists(equipment, x -> lower(x) rlike '{"|.join(keywords)}')").cast('int')
    
    # Check in procedure array
    procedure_check = F.expr(f"exists(procedure, x -> lower(x) rlike '{"|.join(keywords)}')").cast('int')
    
    # Check in description
    description_check = F.when(
        F.lower(F.col('description')).rlike('|'.join(keywords)), 1
    ).otherwise(0)
    
    # Check in specialties
    specialty_check = F.expr(f"exists(specialties, x -> lower(x) rlike '{"|.join(keywords)}')").cast('int')
    
    # Total evidence count (0-5)
    evidence_count = capability_check + equipment_check + procedure_check + description_check + specialty_check
    
    # Convert to trust score (0-3)
    trust_score = F.when(evidence_count >= 3, 3) \
                   .when(evidence_count == 2, 2) \
                   .when(evidence_count == 1, 1) \
                   .otherwise(0)
    
    return trust_score, evidence_count

# Define keyword patterns for each capability
capability_keywords = {
    'emergency': ['emergency', 'trauma', 'accident', 'casualty', '24/7', '24 hour'],
    'maternity': ['maternity', 'obstetric', 'delivery', 'labor', 'prenatal', 'antenatal', 'postnatal', 'gynec'],
    'surgery': ['surgery', 'surgical', 'operation theater', 'ot', 'operative'],
    'icu': ['icu', 'intensive care', 'critical care', 'ccu', 'nicu', 'picu'],
    'dialysis': ['dialysis', 'hemodialysis', 'haemodialysis', 'renal'],
    'nicu': ['nicu', 'neonatal', 'newborn', 'premature'],
    'oncology': ['oncology', 'cancer', 'chemotherapy', 'radiation', 'tumor'],
    'trauma': ['trauma', 'accident', 'injury', 'emergency'],
    'blood_bank': ['blood bank', 'blood transfusion', 'blood storage', 'blood donation'],
    'ambulance': ['ambulance', 'emergency transport', 'patient transport']
}

# Add capability-specific trust scores
facilities_capability_trust = facilities_with_trust

for cap, keywords in capability_keywords.items():
    trust_col, evidence_col = create_capability_check(cap, keywords)
    facilities_capability_trust = facilities_capability_trust \
        .withColumn(f'{cap}_trust_score', trust_col) \
        .withColumn(f'{cap}_evidence_count', evidence_col)

print("Capability-specific trust scores added.")
print("\nSample with capability scores:")
display(
    facilities_capability_trust.select(
        'unique_id', 'name', 'state', 'district',
        'emergency_trust_score', 'maternity_trust_score', 'surgery_trust_score', 
        'icu_trust_score', 'dialysis_trust_score'
    ).limit(10)
)

In [0]:
# Aggregate facilities by state with trust-weighted metrics
state_aggregation = facilities_capability_trust.groupBy('state').agg(
    F.count('*').alias('total_facilities'),
    F.countDistinct('district').alias('districts'),
    F.countDistinct('city').alias('cities'),
    
    # Trust-weighted facility counts by capability
    F.sum(F.when(F.col('emergency_trust_score') >= 2, 1).otherwise(0)).alias('emergency_facilities_trusted'),
    F.sum(F.when(F.col('maternity_trust_score') >= 2, 1).otherwise(0)).alias('maternity_facilities_trusted'),
    F.sum(F.when(F.col('surgery_trust_score') >= 2, 1).otherwise(0)).alias('surgery_facilities_trusted'),
    F.sum(F.when(F.col('icu_trust_score') >= 2, 1).otherwise(0)).alias('icu_facilities_trusted'),
    F.sum(F.when(F.col('dialysis_trust_score') >= 2, 1).otherwise(0)).alias('dialysis_facilities_trusted'),
    F.sum(F.when(F.col('nicu_trust_score') >= 2, 1).otherwise(0)).alias('nicu_facilities_trusted'),
    
    # Overall data quality
    F.avg('overall_trust_score').alias('avg_trust_score'),
    F.sum(F.when(F.col('overall_trust_score') >= 2, 1).otherwise(0)).alias('high_quality_facilities'),
    
    # Capacity indicators
    F.sum(F.coalesce(F.col('capacity'), F.lit(0))).alias('total_capacity'),
    F.sum(F.coalesce(F.col('number_doctors'), F.lit(0))).alias('total_doctors')
).orderBy(F.desc('total_facilities'))

print("State-level aggregation:")
display(state_aggregation)

In [0]:
# Aggregate by district (most granular level for gap analysis)
district_aggregation = facilities_capability_trust \
    .filter(F.col('district').isNotNull()) \
    .groupBy('state', 'district').agg(
        F.count('*').alias('total_facilities'),
        
        # Trust-weighted capability coverage
        F.sum(F.when(F.col('emergency_trust_score') >= 2, 1).otherwise(0)).alias('emergency_facilities'),
        F.sum(F.when(F.col('maternity_trust_score') >= 2, 1).otherwise(0)).alias('maternity_facilities'),
        F.sum(F.when(F.col('surgery_trust_score') >= 2, 1).otherwise(0)).alias('surgery_facilities'),
        F.sum(F.when(F.col('icu_trust_score') >= 2, 1).otherwise(0)).alias('icu_facilities'),
        F.sum(F.when(F.col('dialysis_trust_score') >= 2, 1).otherwise(0)).alias('dialysis_facilities'),
        F.sum(F.when(F.col('nicu_trust_score') >= 2, 1).otherwise(0)).alias('nicu_facilities'),
        
        # Data quality indicators
        F.avg('overall_trust_score').alias('avg_trust_score'),
        F.sum(F.when(F.col('overall_trust_score') >= 2, 1).otherwise(0)).alias('high_quality_facilities'),
        
        # Health context (from NFHS-5)
        F.first('institutional_birth_5y_pct').alias('institutional_birth_pct'),
        F.first('child_12_23m_fully_vaccinated_based_on_information_from_eit_pct').alias('child_vaccination_pct'),
        F.first('hh_member_covered_health_insurance_pct').alias('health_insurance_pct')
    ) \
    .withColumn(
        # Calculate gap risk score (0-100)
        # Lower score = higher risk (few facilities + low data quality + poor health indicators)
        'coverage_score',
        F.when(
            (F.col('total_facilities') > 10) & (F.col('avg_trust_score') >= 2.0), 100
        ).when(
            (F.col('total_facilities') >= 5) & (F.col('avg_trust_score') >= 1.5), 75
        ).when(
            (F.col('total_facilities') >= 2) & (F.col('avg_trust_score') >= 1.0), 50
        ).when(
            F.col('total_facilities') >= 1, 25
        ).otherwise(0)
    ) \
    .withColumn(
        # Classify as medical desert or data-poor region
        'classification',
        F.when(
            (F.col('total_facilities') < 3) & (F.col('avg_trust_score') >= 2.0),
            'High-Confidence Gap (Medical Desert)'
        ).when(
            (F.col('total_facilities') < 5) & (F.col('avg_trust_score') < 1.5),
            'Data-Poor Region (Uncertain)'
        ).when(
            (F.col('total_facilities') >= 5) & (F.col('avg_trust_score') >= 2.0),
            'Adequate Coverage'
        ).otherwise('Moderate Coverage')
    ) \
    .orderBy(F.asc('coverage_score'), F.desc('avg_trust_score'))

print("District-level aggregation with gap classification:")
print(f"Total districts analyzed: {district_aggregation.count():,}")
display(district_aggregation.limit(20))

In [0]:
# Identify high-risk medical deserts (high-confidence gaps)
medical_deserts = district_aggregation.filter(
    F.col('classification') == 'High-Confidence Gap (Medical Desert)'
).orderBy(F.asc('total_facilities'))

print(f"High-Confidence Medical Deserts: {medical_deserts.count():,} districts")
print("\nTop medical desert districts by capability gap:")

# Show districts with specific capability gaps
for capability in ['emergency', 'maternity', 'dialysis', 'icu']:
    cap_deserts = medical_deserts.filter(F.col(f'{capability}_facilities') == 0)
    print(f"\nDistricts with NO {capability} facilities: {cap_deserts.count()}")
    display(cap_deserts.select('state', 'district', 'total_facilities', 'avg_trust_score', 
                                f'{capability}_facilities').limit(10))

In [0]:
# Save the cleaned and enriched facility-level data
facilities_capability_trust.write \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace.default.facilities_cleaned_enriched')

print("Saved: workspace.default.facilities_cleaned_enriched")

# Save state-level aggregation
state_aggregation.write \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace.default.state_aggregation')

print("Saved: workspace.default.state_aggregation")

# Save district-level aggregation
district_aggregation.write \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace.default.district_aggregation')

print("Saved: workspace.default.district_aggregation")

# Save medical deserts list
medical_deserts.write \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace.default.medical_deserts')

print("Saved: workspace.default.medical_deserts")

print("\n✅ All datasets saved successfully!")

## Building the Track 2: Medical Desert Planner App

### Core App Features (Requirements)

**1. Geographic + Capability Selector**
* Select capability (emergency, maternity, dialysis, ICU, NICU, surgery, etc.)
* Select geography level (state, district, city, or PIN code)
* Filter by specific location

**2. Regional Coverage Dashboard**
* Heatmap showing trust-weighted facility coverage
* Color-coded: red (medical desert/high-confidence gap), yellow (data-poor/uncertain), green (adequate coverage)
* Metrics: total facilities, high-trust facilities, capability-specific counts
* Coverage score (0-100) per region

**3. Facility Drill-Down**
* Click on a region to see underlying facilities
* For each facility:
  - Name, location, coordinates
  - Overall trust score (0-3 with explanation)
  - Capability-specific trust scores with **evidence citations** (descriptions, equipment, procedures)
  - Link to source URLs
* **Critical**: Show actual text evidence that supports or contradicts capability claims

**4. Uncertainty Communication**
* Clearly distinguish "few facilities found" vs "little data available"
* Show data completeness metrics per region
* Flag suspicious or conflicting evidence
* Trust score breakdown by evidence type

**5. Planning Scenario Persistence**
* Save user notes and overrides ("This hospital confirmed via phone call", "This district prioritized for Q3")
* Create named scenarios ("Q3 2026 Planning", "Emergency Care Expansion")
* Track planning decisions per region/facility
* Export scenarios to CSV/PDF

### Technical Implementation

**Data Layer** (Already Complete!):
* ✅ `workspace.default.facilities_cleaned_enriched` - Cleaned facility data with trust scores
* ✅ `workspace.default.state_aggregation` - State-level metrics
* ✅ `workspace.default.district_aggregation` - District-level metrics with gap classifications
* ✅ `workspace.default.medical_deserts` - High-confidence medical desert regions

**App Framework Options**:
1. **Databricks Apps (Recommended)**: Uses Streamlit/Gradio, runs on Databricks serverless compute
2. **Standalone Dashboard**: Using Lakeview Dashboards with SQL queries
3. **Custom Web App**: Flask/FastAPI with Databricks SQL connector

**App Structure** (using Databricks Apps with Streamlit):
```python
import streamlit as st
import pandas as pd
from databricks import sql
import plotly.express as px

# Sidebar: Filters
capability = st.sidebar.selectbox('Capability', ['emergency', 'maternity', 'dialysis', ...])
geography_level = st.sidebar.selectbox('Geography', ['State', 'District', 'City'])

# Main: Coverage Heatmap
# Query district_aggregation, filter by capability, show map

# Drill-down: Facility List
# Show facilities for selected region with evidence

# Scenario Management
# Save/load user notes using Delta table
```

### Quick Start Commands

```bash
# Create app directory
dbfs mkdirs /Workspace/Users/<your-email>/medical-desert-app

# Create app.py with Streamlit code
# Create app.yaml with configuration

# Deploy app
databricks apps create medical-desert-planner \
  --source-code-path /Workspace/Users/<your-email>/medical-desert-app
```

### Key Differentiators for Track 2

* **Focus on aggregation**: Show regional patterns, not just individual facilities
* **Uncertainty handling**: Distinguish real gaps from data gaps
* **Health context**: Include NFHS-5 indicators to contextualize gaps
* **Scenario planning**: Support multi-round decision-making workflows
* **Evidence grounding**: Every metric links back to facility-level evidence

## Pipeline Summary

### Outputs Created:
1. **`workspace.default.facilities_cleaned_enriched`** - Cleaned facility records with:
   - Parsed arrays (specialties, capability, equipment, procedure)
   - Validated coordinates
   - District mappings from pincode directory
   - NFHS-5 health indicators
   - Overall and capability-specific trust scores (0-3 scale)

2. **`workspace.default.state_aggregation`** - State-level metrics

3. **`workspace.default.district_aggregation`** - District-level metrics with:
   - Trust-weighted facility counts by capability
   - Coverage scores and gap classifications
   - Health context indicators

4. **`workspace.default.medical_deserts`** - High-confidence medical desert districts

### Key Findings:
- **Data Quality**: Variable completeness across evidence fields
- **Trust Scoring**: Facilities rated 0-3 based on evidence completeness
- **Geographic Gaps**: Districts classified as medical deserts vs data-poor regions
- **Capability Gaps**: Specific gaps in emergency, maternity, dialysis, ICU, NICU services

### Next Steps for App Development:
1. **Backend**: Use the cleaned tables as data source
2. **UI Components**:
   - Geographic selector (state → district → city/pincode)
   - Capability selector (emergency, maternity, etc.)
   - Facility drill-down with evidence citations
   - Scenario saving (user notes, overrides, planning decisions)
3. **Features**:
   - Interactive map visualization
   - Trust-weighted coverage heatmaps
   - Facility detail panels with evidence highlighting
   - Planning scenario management (save/load/export)
4. **Databricks App**: Build UI using Streamlit or Dash